# Fusion swissALTI3D + swissBATHY3D → DEM unifié (EPSG:2056)

Ce notebook :
1. Télécharge les tuiles **swissALTI3D** et **swissBATHY3D** via l'API STAC de swisstopo pour une emprise donnée
2. Vérifie et aligne les systèmes de coordonnées (cible : **EPSG:2056** — CH1903+/LV95)
3. Reprojette la bathymétrie si nécessaire
4. Fusionne les deux modèles en un seul DEM (la bathymétrie remplace l'alti dans les zones lacustres)
5. Sauvegarde le résultat en GeoTIFF
6. Affiche une visualisation rapide (ombrage, profil)
7. Exporte le DEM en format STL pour impression 3D

---
**Source** : [swisstopo Open Data](https://www.swisstopo.admin.ch)  
**Licence** : Open use, attribution obligatoire (© swisstopo)


## 0. Installation des dépendances

In [ ]:
# Exécuter une seule fois pour installer les paquets nécessaires
# !pip install rasterio pyproj requests numpy matplotlib numpy-stl tqdm

## 1. Imports et configuration

In [ ]:
import os
import json
import math
import tempfile
import warnings
from pathlib import Path

import numpy as np
import requests
import rasterio
from rasterio.crs import CRS
from rasterio.enums import Resampling
from rasterio.merge import merge
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject
from rasterio.mask import mask as rio_mask
from pyproj import Transformer
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LightSource
from stl import mesh as stl_mesh  # numpy-stl

warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

print("✅ Imports OK")

## 2. Paramètres utilisateur

Définissez ici votre **emprise en EPSG:2056** (coordonnées LV95, en mètres)  
et le répertoire de sortie.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EMPRISE EN EPSG:2056  (LV95 — mètres)
# Exemple : rive nord-est du Lac Léman autour de Lausanne
# ─────────────────────────────────────────────────────────────────────────────
BBOX_2056 = {
    "xmin": 2530000,   # Est min
    "ymin": 1149000,   # Nord min
    "xmax": 2537000,   # Est max
    "ymax": 1155000,   # Nord max
}

# Résolution cible en mètres (2 m = résolution native swissALTI3D)
TARGET_RESOLUTION = 2.0

# Répertoire de sortie
OUTPUT_DIR = Path("./output_dem")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# CRS cibles
# ─────────────────────────────────────────────────────────────────────────────
TARGET_CRS = CRS.from_epsg(2056)   # CH1903+ / LV95
WGS84_CRS  = CRS.from_epsg(4326)   # WGS 84 (requis par l'API STAC)

# ─────────────────────────────────────────────────────────────────────────────
# Identifiants des collections STAC swisstopo
# ─────────────────────────────────────────────────────────────────────────────
STAC_BASE   = "https://data.geo.admin.ch/api/stac/v0.9"
COL_ALTI    = "ch.swisstopo.swissalti3d"
COL_BATHY   = "ch.swisstopo.swissbathy3d"

print(f"Emprise LV95 : E {BBOX_2056['xmin']}–{BBOX_2056['xmax']} / N {BBOX_2056['ymin']}–{BBOX_2056['ymax']}")
print(f"Sortie       : {OUTPUT_DIR.resolve()}")

## 3. Conversion de l'emprise LV95 → WGS84 (pour l'API STAC)

In [ ]:
def lv95_bbox_to_wgs84(bbox: dict) -> dict:
    """Convertit une emprise EPSG:2056 en EPSG:4326 (lon/lat)."""
    transformer = Transformer.from_crs("EPSG:2056", "EPSG:4326", always_xy=True)
    lon_min, lat_min = transformer.transform(bbox["xmin"], bbox["ymin"])
    lon_max, lat_max = transformer.transform(bbox["xmax"], bbox["ymax"])
    return {"lon_min": lon_min, "lat_min": lat_min,
            "lon_max": lon_max, "lat_max": lat_max}

BBOX_WGS84 = lv95_bbox_to_wgs84(BBOX_2056)
print("Emprise WGS84 :")
print(f"  lon {BBOX_WGS84['lon_min']:.6f} → {BBOX_WGS84['lon_max']:.6f}")
print(f"  lat {BBOX_WGS84['lat_min']:.6f} → {BBOX_WGS84['lat_max']:.6f}")

## 4. Fonctions utilitaires : API STAC swisstopo

In [ ]:
def stac_get_items(collection: str, bbox_wgs84: dict, limit: int = 100) -> list:
    """
    Interroge l'API STAC pour récupérer les items d'une collection
    intersectant l'emprise WGS84 donnée.
    Gère la pagination automatiquement.
    """
    bbox_str = (
        f"{bbox_wgs84['lon_min']},{bbox_wgs84['lat_min']},"
        f"{bbox_wgs84['lon_max']},{bbox_wgs84['lat_max']}"
    )
    url = f"{STAC_BASE}/collections/{collection}/items"
    params = {"bbox": bbox_str, "limit": limit}

    items = []
    while url:
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()
        items.extend(data.get("features", []))
        # Pagination : lien "next"
        next_link = next((lk["href"] for lk in data.get("links", [])
                          if lk.get("rel") == "next"), None)
        url = next_link
        params = {}  # paramètres déjà encodés dans le lien next
    return items


def best_asset_href(item: dict, preferred_crs: str = "2056",
                    preferred_res: float = 2.0) -> str | None:
    """
    Choisit parmi les assets d'un item STAC le GeoTIFF le plus adapté :
    priorité à EPSG:2056, résolution 2 m.
    """
    candidates = []
    for key, asset in item.get("assets", {}).items():
        href = asset.get("href", "")
        if not href.endswith(".tif"):
            continue
        score = 0
        if preferred_crs in href:
            score += 10
        # Résolution encodée dans le nom de fichier (ex. "_2.0_" ou "_2_")
        for res_str in [f"_{preferred_res}_", f"_{int(preferred_res)}_"]:
            if res_str in href:
                score += 5
        candidates.append((score, href))

    if not candidates:
        return None
    candidates.sort(key=lambda x: -x[0])
    return candidates[0][1]


def download_file(url: str, dest: Path, session: requests.Session) -> Path:
    """Télécharge un fichier avec barre de progression."""
    if dest.exists():
        return dest  # déjà téléchargé
    with session.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(
            desc=dest.name, total=total, unit="B",
            unit_scale=True, unit_divisor=1024, leave=False
        ) as bar:
            for chunk in r.iter_content(chunk_size=65536):
                f.write(chunk)
                bar.update(len(chunk))
    return dest

print("✅ Fonctions STAC définies")

## 5. Téléchargement des tuiles swissALTI3D et swissBATHY3D

In [ ]:
def fetch_tiles(collection: str, label: str, out_subdir: Path) -> list[Path]:
    """Interroge l'API STAC et télécharge toutes les tuiles pour l'emprise."""
    out_subdir.mkdir(parents=True, exist_ok=True)

    print(f"\n🔍 Recherche des tuiles {label}…")
    items = stac_get_items(collection, BBOX_WGS84)
    print(f"   {len(items)} item(s) trouvé(s) dans l'emprise")

    if not items:
        print(f"   ⚠️  Aucune donnée {label} dans cette emprise.")
        return []

    paths = []
    session = requests.Session()
    for item in tqdm(items, desc=f"Téléchargement {label}"):
        href = best_asset_href(item)
        if href is None:
            print(f"   ⚠️  Aucun asset GeoTIFF pour {item['id']} — ignoré")
            continue
        fname = Path(href).name
        dest  = out_subdir / fname
        download_file(href, dest, session)
        paths.append(dest)

    print(f"   ✅ {len(paths)} fichier(s) téléchargé(s) dans {out_subdir}")
    return paths


# ── Téléchargement ──────────────────────────────────────────────────────────
alti_files  = fetch_tiles(COL_ALTI,  "swissALTI3D",  OUTPUT_DIR / "alti_tiles")
bathy_files = fetch_tiles(COL_BATHY, "swissBATHY3D", OUTPUT_DIR / "bathy_tiles")

print(f"\nRésumé : {len(alti_files)} tuile(s) ALTI  |  {len(bathy_files)} tuile(s) BATHY")

## 6. Vérification du système de coordonnées

In [ ]:
def check_crs(files: list[Path], label: str) -> CRS | None:
    """Inspecte le CRS de chaque fichier et retourne le CRS détecté."""
    if not files:
        print(f"  [{label}] Aucun fichier à vérifier.")
        return None
    crss = set()
    for f in files:
        with rasterio.open(f) as src:
            crs = src.crs
            crss.add(crs.to_epsg())
    if len(crss) == 1:
        epsg = list(crss)[0]
        match = "✅" if epsg == 2056 else "⚠️ "
        print(f"  [{label}] CRS uniforme : EPSG:{epsg} {match}")
        return CRS.from_epsg(epsg)
    else:
        print(f"  [{label}] ⚠️  Plusieurs CRS détectés : {crss} — reprojection nécessaire")
        return None


print("─── Vérification CRS ───────────────────────────────────────")
alti_crs  = check_crs(alti_files,  "swissALTI3D")
bathy_crs = check_crs(bathy_files, "swissBATHY3D")

need_reproject_alti  = alti_crs  is not None and alti_crs.to_epsg()  != 2056
need_reproject_bathy = bathy_crs is not None and bathy_crs.to_epsg() != 2056

if need_reproject_alti:
    print("  → swissALTI3D doit être reprojeté en EPSG:2056")
if need_reproject_bathy:
    print("  → swissBATHY3D doit être reprojeté en EPSG:2056")
if not need_reproject_alti and not need_reproject_bathy:
    print("  ✅ Aucune reprojection nécessaire")

## 7. Reprojection si nécessaire

In [ ]:
def reproject_to_lv95(src_path: Path, dst_dir: Path) -> Path:
    """Reprojette un raster vers EPSG:2056 et sauvegarde le résultat."""
    dst_path = dst_dir / (src_path.stem + "_2056.tif")
    if dst_path.exists():
        return dst_path

    with rasterio.open(src_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, TARGET_CRS, src.width, src.height, *src.bounds
        )
        kwargs = src.meta.copy()
        kwargs.update({
            "crs": TARGET_CRS,
            "transform": transform,
            "width": width,
            "height": height,
            "compress": "deflate",
        })
        with rasterio.open(dst_path, "w", **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=TARGET_CRS,
                    resampling=Resampling.bilinear,
                )
    return dst_path


reproj_dir = OUTPUT_DIR / "reprojected"
reproj_dir.mkdir(exist_ok=True)

if need_reproject_alti:
    print("Reprojection swissALTI3D…")
    alti_files = [reproject_to_lv95(f, reproj_dir) for f in tqdm(alti_files)]

if need_reproject_bathy:
    print("Reprojection swissBATHY3D…")
    bathy_files = [reproject_to_lv95(f, reproj_dir) for f in tqdm(bathy_files)]

print("✅ Fichiers prêts en EPSG:2056")

## 8. Fusion des tuiles par produit (mosaïque) et découpe sur l'emprise

In [ ]:
from shapely.geometry import box
import geopandas as gpd

def mosaic_and_clip(files: list[Path], out_path: Path) -> Path:
    """
    Fusionne plusieurs tuiles GeoTIFF en une seule mosaïque,
    puis la découpe sur l'emprise BBOX_2056.
    """
    if not files:
        return None

    datasets = [rasterio.open(f) for f in files]
    mosaic_data, mosaic_transform = merge(datasets)
    meta = datasets[0].meta.copy()
    meta.update({
        "driver": "GTiff",
        "height": mosaic_data.shape[1],
        "width":  mosaic_data.shape[2],
        "transform": mosaic_transform,
        "compress": "deflate",
        "crs": TARGET_CRS,
    })
    for ds in datasets:
        ds.close()

    # Écriture temporaire de la mosaïque
    tmp_mosaic = out_path.parent / (out_path.stem + "_full.tif")
    with rasterio.open(tmp_mosaic, "w", **meta) as dst:
        dst.write(mosaic_data)

    # Découpe sur l'emprise
    bbox_geom = box(
        BBOX_2056["xmin"], BBOX_2056["ymin"],
        BBOX_2056["xmax"], BBOX_2056["ymax"]
    )
    with rasterio.open(tmp_mosaic) as src:
        clipped, clip_transform = rio_mask(src, [bbox_geom], crop=True)
        clip_meta = src.meta.copy()
        clip_meta.update({
            "height": clipped.shape[1],
            "width":  clipped.shape[2],
            "transform": clip_transform,
        })
        with rasterio.open(out_path, "w", **clip_meta) as dst:
            dst.write(clipped)

    tmp_mosaic.unlink()  # nettoyage
    print(f"  ✅ {out_path.name} — {clipped.shape[2]}×{clipped.shape[1]} px")
    return out_path


print("─── Mosaïquage + découpe ────────────────────────────────────")
alti_mosaic_path  = OUTPUT_DIR / "swissalti3d_mosaic.tif"
bathy_mosaic_path = OUTPUT_DIR / "swissbathy3d_mosaic.tif"

alti_mosaic  = mosaic_and_clip(alti_files,  alti_mosaic_path)  if alti_files  else None
bathy_mosaic = mosaic_and_clip(bathy_files, bathy_mosaic_path) if bathy_files else None

## 9. Fusion ALTI + BATHY → DEM unifié

Stratégie : là où la bathymétrie fournit des données valides (valeur ≠ nodata),
elle remplace l'altitude de swissALTI3D.  
Dans swissALTI3D, les surfaces lacustres sont généralement encodées à l'altitude du plan d'eau (valeur constante). La bathymétrie donne la topographie du fond.

In [ ]:
def resample_to_reference(src_path: Path, ref_path: Path, out_path: Path) -> Path:
    """
    Rééchantillonne src_path pour correspondre exactement à la grille
    (résolution, extent, CRS) de ref_path.
    """
    with rasterio.open(ref_path) as ref:
        ref_meta = ref.meta.copy()
        ref_transform = ref.transform
        ref_crs = ref.crs
        ref_shape = (ref.height, ref.width)

    with rasterio.open(src_path) as src:
        data = np.empty(ref_shape, dtype=ref_meta["dtype"])
        reproject(
            source=rasterio.band(src, 1),
            destination=data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.bilinear,
        )
        src_nodata = src.nodata

    ref_meta.update({"compress": "deflate"})
    with rasterio.open(out_path, "w", **ref_meta) as dst:
        dst.write(data, 1)
        if src_nodata is not None:
            dst.update_tags(1, nodata=src_nodata)
    return out_path


def merge_alti_bathy(alti_path: Path, bathy_path: Path | None,
                     out_path: Path) -> Path:
    """
    Fusionne swissALTI3D et swissBATHY3D :
    - la bathymétrie (valeurs négatives = fond lacustre) remplace l'ALTI
      dans les zones où elle est définie.
    - le résultat est en EPSG:2056, résolution de l'ALTI.
    """
    with rasterio.open(alti_path) as src_alti:
        alti_data  = src_alti.read(1).astype(np.float32)
        alti_nd    = src_alti.nodata if src_alti.nodata is not None else -9999
        out_meta   = src_alti.meta.copy()

    if bathy_path is None:
        print("  ⚠️  Pas de données bathymétriques — DEM = swissALTI3D seul")
        import shutil; shutil.copy(alti_path, out_path)
        return out_path

    # Rééchantillonnage de la bathymétrie sur la grille ALTI
    bathy_resampled = out_path.parent / "bathy_resampled.tif"
    resample_to_reference(bathy_path, alti_path, bathy_resampled)

    with rasterio.open(bathy_resampled) as src_bathy:
        bathy_data = src_bathy.read(1).astype(np.float32)
        bathy_nd   = src_bathy.nodata if src_bathy.nodata is not None else -9999

    # Masque : pixels valides de la bathymétrie
    bathy_valid = (bathy_data != bathy_nd) & np.isfinite(bathy_data)

    fused = alti_data.copy()
    fused[bathy_valid] = bathy_data[bathy_valid]

    out_meta.update({
        "dtype": "float32",
        "nodata": alti_nd,
        "compress": "deflate",
    })
    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(fused, 1)

    n_bathy_px = int(bathy_valid.sum())
    print(f"  Pixels bathymétriques intégrés : {n_bathy_px:,} "
          f"({100*n_bathy_px/fused.size:.1f}% de la grille)")
    print(f"  Altitude min/max DEM fusionné : "
          f"{fused[fused != alti_nd].min():.1f} m / {fused[fused != alti_nd].max():.1f} m")
    return out_path


DEM_FUSED_PATH = OUTPUT_DIR / "dem_fused_2056.tif"

print("─── Fusion ALTI + BATHY ────────────────────────────────────")
merge_alti_bathy(alti_mosaic, bathy_mosaic, DEM_FUSED_PATH)
print(f"  ✅ DEM fusionné sauvegardé : {DEM_FUSED_PATH}")

## 10. Visualisation rapide du DEM fusionné

In [ ]:
def visualize_dem(dem_path: Path):
    """Affiche le DEM avec ombrage, histogramme et profil transversal."""
    with rasterio.open(dem_path) as src:
        dem = src.read(1).astype(np.float32)
        nodata = src.nodata if src.nodata is not None else -9999
        transform = src.transform
        bounds = src.bounds
        res_m = abs(transform.a)

    # Masquer nodata
    dem_ma = np.ma.masked_where(dem == nodata, dem)

    # Ombrage (hillshade)
    ls = LightSource(azdeg=315, altdeg=45)
    hs = ls.hillshade(dem_ma.filled(dem_ma.mean()), vert_exag=3,
                     dx=res_m, dy=res_m)

    fig = plt.figure(figsize=(16, 12))
    fig.suptitle("DEM fusionné swissALTI3D + swissBATHY3D (EPSG:2056)",
                 fontsize=14, fontweight="bold")

    # ── Panneau 1 : ombrage + couleurs ────────────────────────────────────
    ax1 = fig.add_subplot(2, 2, 1)
    extent_m = [bounds.left, bounds.right, bounds.bottom, bounds.top]
    im = ax1.imshow(dem_ma, cmap="terrain", origin="upper",
                    extent=extent_m, interpolation="bilinear")
    ax1.imshow(hs, cmap="gray", alpha=0.4, origin="upper",
               extent=extent_m, interpolation="bilinear")
    plt.colorbar(im, ax=ax1, label="Altitude (m)", shrink=0.8)
    ax1.set_title("DEM coloré + ombrage")
    ax1.set_xlabel("E (m LV95)"); ax1.set_ylabel("N (m LV95)")
    ax1.ticklabel_format(style="plain", axis="both")

    # ── Panneau 2 : ombrage seul ───────────────────────────────────────────
    ax2 = fig.add_subplot(2, 2, 2)
    ax2.imshow(hs, cmap="gray", origin="upper",
               extent=extent_m, interpolation="bilinear")
    ax2.set_title("Ombrage (hillshade)")
    ax2.set_xlabel("E (m LV95)"); ax2.set_ylabel("N (m LV95)")
    ax2.ticklabel_format(style="plain", axis="both")

    # ── Panneau 3 : histogramme ───────────────────────────────────────────
    ax3 = fig.add_subplot(2, 2, 3)
    valid = dem_ma.compressed()
    ax3.hist(valid, bins=100, color="steelblue", edgecolor="none", alpha=0.85)
    ax3.axvline(0, color="red", linestyle="--", label="0 m (niveau mer)")
    ax3.set_xlabel("Altitude (m)")
    ax3.set_ylabel("Nombre de pixels")
    ax3.set_title("Distribution des altitudes")
    ax3.legend()

    # ── Panneau 4 : profil transversal central ────────────────────────────
    ax4 = fig.add_subplot(2, 2, 4)
    mid_row = dem.shape[0] // 2
    profile = dem_ma[mid_row, :]
    x_coords = np.linspace(bounds.left, bounds.right, dem.shape[1])
    ax4.plot(x_coords, profile, color="saddlebrown", linewidth=1)
    ax4.fill_between(x_coords, profile.min() - 10, profile,
                     where=profile > 0, color="tan", alpha=0.5, label="Terrain")
    ax4.fill_between(x_coords, profile.min() - 10, profile,
                     where=profile <= 0, color="steelblue", alpha=0.5, label="Fond lacustre")
    ax4.axhline(0, color="blue", linestyle="--", linewidth=0.7, label="0 m")
    ax4.set_xlabel("E (m LV95)")
    ax4.set_ylabel("Altitude (m)")
    ax4.set_title(f"Profil transversal (N={bounds.bottom + (bounds.top - bounds.bottom)/2:.0f} m)")
    ax4.legend(fontsize=8)

    plt.tight_layout()
    viz_path = OUTPUT_DIR / "dem_visualisation.png"
    plt.savefig(viz_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  ✅ Visualisation sauvegardée : {viz_path}")


visualize_dem(DEM_FUSED_PATH)

## 11. Export au format STL (pour impression 3D ou visualisation 3D)

In [ ]:
def dem_to_stl(
    dem_path: Path,
    stl_path: Path,
    z_scale: float = 1.0,
    max_size_px: int = 500,
    base_depth: float = 50.0,
):
    """
    Convertit un DEM GeoTIFF en fichier STL.

    Paramètres
    ----------
    dem_path    : chemin vers le GeoTIFF source
    stl_path    : chemin de sortie .stl
    z_scale     : exagération verticale (ex. 2.0 = ×2)
    max_size_px : résolution maximale (le DEM est downsampléé si nécessaire)
    base_depth  : épaisseur de la base solide (en mètres)

    Notes
    -----
    - Les coordonnées XY sont en mètres (LV95).
    - L'axe Z est l'altitude en mètres, exagérée par z_scale.
    - Une base plate est ajoutée sous le modèle pour le rendre imprimable.
    """
    with rasterio.open(dem_path) as src:
        # Réduit la résolution si nécessaire pour limiter la taille du STL
        scale = max(src.width, src.height) / max_size_px
        if scale > 1:
            new_w = int(src.width  / scale)
            new_h = int(src.height / scale)
            dem = src.read(
                1,
                out_shape=(new_h, new_w),
                resampling=Resampling.bilinear,
            ).astype(np.float32)
            res_x = (src.bounds.right  - src.bounds.left)   / new_w
            res_y = (src.bounds.top    - src.bounds.bottom)  / new_h
            x0, y0 = src.bounds.left, src.bounds.bottom
        else:
            dem  = src.read(1).astype(np.float32)
            res_x = abs(src.transform.a)
            res_y = abs(src.transform.e)
            x0, y0 = src.bounds.left, src.bounds.bottom

        nodata = src.nodata if src.nodata is not None else -9999

    rows, cols = dem.shape
    print(f"  Grille STL : {cols}×{rows} px (résolution ~{res_x:.1f}×{res_y:.1f} m)")

    # Remplacement des nodata par l'altitude minimale
    dem[dem == nodata] = np.nan
    z_min = float(np.nanmin(dem))
    dem = np.where(np.isnan(dem), z_min, dem)

    # Application du facteur d'exagération
    z_base = z_min - base_depth / z_scale

    # Grilles de coordonnées XYZ
    xs = x0 + np.arange(cols) * res_x + res_x / 2
    ys = y0 + np.arange(rows - 1, -1, -1) * res_y + res_y / 2  # flip Y
    X, Y = np.meshgrid(xs, ys)
    Z = dem * z_scale

    # ── Construction des triangles ─────────────────────────────────────────
    # Surface supérieure : 2 triangles par quad
    def quad_to_triangles(x0q, y0q, x1q, y1q, z00, z10, z01, z11):
        v0 = [x0q, y0q, z00]
        v1 = [x1q, y0q, z10]
        v2 = [x0q, y1q, z01]
        v3 = [x1q, y1q, z11]
        return [(v0, v1, v2), (v1, v3, v2)]

    top_triangles = []
    for r in range(rows - 1):
        for c in range(cols - 1):
            tris = quad_to_triangles(
                X[r, c], Y[r, c], X[r, c+1], Y[r+1, c],
                Z[r, c], Z[r, c+1], Z[r+1, c], Z[r+1, c+1]
            )
            top_triangles.extend(tris)

    # Base plate
    z_b = z_base * z_scale
    corners = [
        [X[0, 0],      Y[0, 0],      z_b],
        [X[0, -1],     Y[0, -1],     z_b],
        [X[-1, 0],     Y[-1, 0],     z_b],
        [X[-1, -1],    Y[-1, -1],    z_b],
    ]
    base_triangles = [
        (corners[0], corners[2], corners[1]),
        (corners[1], corners[2], corners[3]),
    ]

    # Faces latérales (4 côtés)
    def side_strip(edge_r, edge_c):
        tris = []
        for i in range(len(edge_r) - 1):
            x0s, y0s, z0s = X[edge_r[i], edge_c[i]], Y[edge_r[i], edge_c[i]], Z[edge_r[i], edge_c[i]]
            x1s, y1s, z1s = X[edge_r[i+1], edge_c[i+1]], Y[edge_r[i+1], edge_c[i+1]], Z[edge_r[i+1], edge_c[i+1]]
            tris += [
                ([x0s, y0s, z_b], [x1s, y1s, z_b], [x0s, y0s, z0s]),
                ([x1s, y1s, z_b], [x1s, y1s, z1s], [x0s, y0s, z0s]),
            ]
        return tris

    side_triangles = []
    side_triangles += side_strip(np.zeros(cols, int),  np.arange(cols))         # top edge
    side_triangles += side_strip(np.full(cols, rows-1, int), np.arange(cols))    # bottom edge
    side_triangles += side_strip(np.arange(rows), np.zeros(rows, int))           # left edge
    side_triangles += side_strip(np.arange(rows), np.full(rows, cols-1, int))    # right edge

    all_triangles = top_triangles + base_triangles + side_triangles
    n_tri = len(all_triangles)
    print(f"  Triangles générés : {n_tri:,}")

    # ── Création du mesh numpy-stl ─────────────────────────────────────────
    model = stl_mesh.Mesh(np.zeros(n_tri, dtype=stl_mesh.Mesh.dtype))
    for i, (v0, v1, v2) in enumerate(all_triangles):
        model.v0[i] = v0
        model.v1[i] = v1
        model.v2[i] = v2

    model.update_normals()
    model.save(str(stl_path))
    size_mb = stl_path.stat().st_size / 1e6
    print(f"  ✅ STL sauvegardé : {stl_path} ({size_mb:.1f} MB)")


STL_PATH = OUTPUT_DIR / "dem_fused_2056.stl"
print("─── Export STL ─────────────────────────────────────────────")
dem_to_stl(
    dem_path    = DEM_FUSED_PATH,
    stl_path    = STL_PATH,
    z_scale     = 2.0,    # exagération verticale ×2
    max_size_px = 400,    # résolution max pour garder un fichier raisonnable
    base_depth  = 100.0,  # base solide de 100 m sous le point le plus bas
)

## 12. Récapitulatif des fichiers produits

In [ ]:
print("═" * 60)
print("  FICHIERS PRODUITS")
print("═" * 60)
for p in sorted(OUTPUT_DIR.glob("*.tif")) + sorted(OUTPUT_DIR.glob("*.stl")) + sorted(OUTPUT_DIR.glob("*.png")):
    size_mb = p.stat().st_size / 1e6
    print(f"  {p.name:<40} {size_mb:6.1f} MB")
print("─" * 60)
print()
print("📌 Attribution : © swisstopo — Open use, source obligatoire")